# A FAIR&#178; Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")

# Optionally, view available keys
print("\nMetadata keys:")
print(list(metadata.keys()))

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Entities in the dataset are referenced by their `@id`s.**

In [ ]:
# Get all record sets in the dataset
# All entities are referenced by their @id
record_sets = dataset.metadata.record_sets

if record_sets:
    print("Record Sets:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']}  | name: {rs.get('name', '')}")
        # Print fields for each record set
        if 'fields' in rs:
            for f in rs['fields']:
                print(f"    Field @id: {f['@id']} | name: {f.get('name','')} | dataType: {f.get('dataType','')}")
else:
    print("No record sets found in the metadata.")

# If no record_sets were found above (sometimes metadata.record_sets returns empty),
# examine the raw metadata JSON for alternative linking:
if not record_sets:
    if 'recordSet' in metadata:
        if isinstance(metadata['recordSet'], list):
            print('Record Sets (from raw json):')
            for rs in metadata['recordSet']:
                print(f"  @id: {rs}")
        else:
            print(f"  @id: {metadata['recordSet']}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use @id from the prior overview
# For this dataset, record set IDs are not listed in the metadata in full; they may be loaded from the dataset.records method
# If the record set list is not directly available, mlcroissant will often expose data via dataset.records(record_set=...) with the actual record set ID

record_sets = []
# Try to extract all @id for record sets
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
elif 'recordSet' in metadata:
    if isinstance(metadata['recordSet'], list):
        record_sets = [rs for rs in metadata['recordSet']]
    else:
        record_sets = [metadata['recordSet']]
else:
    print("Unable to locate record set @id list in metadata.")

# Create DataFrames for each record set
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Record Set {record_set_id} loaded with shape {df.shape}")
        else:
            print(f"Record Set {record_set_id} has no records.")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Display columns of one DataFrame for inspection
if dataframes:
    main_record_set = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for record set @id: {main_record_set}")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No DataFrames created from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping by attributes to prepare for further analysis.

All column references must use their full `@id` as displayed above.

In [ ]:
# EDA: Filtering, normalization, grouping
import numpy as np

# Select a main record set and a numeric field (@id)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f'Record set being analyzed: {record_set_id}')

    # Check for numeric columns (example: GAD-7 score)
    numeric_candidates = [col for col in df.columns if 'gad_7_score' in col or 'phq_9_score' in col or df[col].dtype in [np.int64, np.float64]]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f'Using numeric field: {numeric_field_id}')
        threshold = 10
        # Filtering
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (for example, gender or education level)
        group_candidates = [col for col in df.columns if 'gender' in col or 'level_of_education' in col]
        group_field = group_candidates[0] if group_candidates else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped average {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found in DataFrame.")
    else:
        print("No numeric fields found for analysis in the DataFrame.")
else:
    print("No DataFrames available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization examples
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_candidates = [col for col in df.columns if 'gad_7_score' in col or 'phq_9_score' in col or df[col].dtype in [np.int64, np.float64]]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        plt.figure(figsize=(8,6))
        sns.histplot(df[numeric_field_id], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        
        # Boxplot by group (e.g., gender or education)
        group_candidates = [col for col in df.columns if 'gender' in col or 'level_of_education' in col]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            plt.figure(figsize=(8,6))
            sns.boxplot(x=df[group_field], y=df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print("No numeric variables for visualization.")
else:
    print("No DataFrames available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, the `mlcroissant` library was used to load metadata and records from the FAIR&#178; Mental Health Survey Dataset from Kilifi County, Kenya, referencing all entities by their `@id`.
* We explored record sets and fields, loaded data into DataFrames, and performed basic EDA (filtering, normalization, grouping) and visualized numeric survey scores.
* The dataset contains rich mental health indicators and demographic information, with potential for deeper assessment of population-level trends and subgroup analysis.

**Next steps:** Further investigate correlations between psychological score fields and demographic variables, address possible missing values, and model prediction/forecasting tasks for mental health outcomes.